# Classifying iris flowers with a decision tree
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/KatherineElkins/programminghumanity2026/blob/main/iris_decision_tree_2026.ipynb)

Can four flower measurements predict the species? We will train a tree, follow its rules, and test it on flowers it has not seen. Adapted from our 2025 Iris activity.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, ConfusionMatrixDisplay

iris = load_iris()
X, y = iris.data, iris.target
flowers = pd.DataFrame(X, columns=iris.feature_names)
flowers['species'] = iris.target_names[y]
display(flowers.head())
print('Features:', ', '.join(iris.feature_names))
print('Labels:', ', '.join(iris.target_names))


## 1. Keep some flowers unseen
We reserve 20% for testing. Both sets contain all three species. The fixed split lets us compare tree depths fairly.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)
print(f'Training: {len(y_train)} flowers; testing: {len(y_test)} flowers')


## 2. Train and read a tree
Start with depth 2. Follow one path: which measurement is tested first, and what happens on each side of the threshold?

`value` shows the training class composition at a node (counts or proportions, depending on the scikit-learn version); `class` is the predicted species.

In [ ]:
max_depth = 2  # Try 1, 2, 3, then None for an unrestricted tree.
model = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
model.fit(X_train, y_train)
fig, ax = plt.subplots(figsize=(15, 8))
plot_tree(model, feature_names=iris.feature_names,
          class_names=list(iris.target_names), filled=True, rounded=True, ax=ax)
plt.show()
print(f'Training accuracy: {model.score(X_train, y_train):.1%}')
print(f'Test accuracy: {model.score(X_test, y_test):.1%}')


## 3. Predict one unseen flower
Before running the cell, use the measurements and tree to predict its species. Then compare your prediction with the model and the recorded label.

In [ ]:
flower_number = 0  # Choose 0–29.
if not 0 <= flower_number < len(X_test):
    raise ValueError(f'Choose a flower number from 0 to {len(X_test)-1}.')
measurements = X_test[flower_number]
display(pd.Series(measurements, index=iris.feature_names, name='Measurement (cm)'))


In [ ]:
prediction = model.predict([measurements])[0]
print('Model prediction:', iris.target_names[prediction])
print('Recorded species:', iris.target_names[y_test[flower_number]])


## 4. Does a deeper tree predict better?
Compare training and test accuracy on the same split. Does increasing depth always help on unseen flowers? A perfect training score does not establish good prediction. This small test set is an illustration; choosing depth from these scores would require a separate validation set or cross-validation for an independent final evaluation.

In [ ]:
results = []
for depth in [1, 2, 3, 4, None]:
    candidate = DecisionTreeClassifier(max_depth=depth, random_state=42)
    candidate.fit(X_train, y_train)
    results.append({'Maximum depth': str(depth) if depth else 'Unrestricted',
                    'Training accuracy': candidate.score(X_train, y_train),
                    'Test accuracy': candidate.score(X_test, y_test)})
display(pd.DataFrame(results).style.format({'Training accuracy': '{:.1%}', 'Test accuracy': '{:.1%}'}))
ConfusionMatrixDisplay.from_estimator(model, X_test, y_test,
    display_labels=iris.target_names, cmap='Blues')
plt.title('Held-out flowers: the tree selected in section 2')
plt.show()


## Discuss
- How is this like classifying SF/NY homes? What are the features and labels in each case?
- What does the tree learn from examples, rather than receive as a rule we write?
- Point to one split and explain how it changes a prediction.

Data: [scikit-learn Iris dataset documentation](https://scikit-learn.org/stable/datasets/toy_dataset.html#iris-plants-dataset).